# Calendar-time six-factor alpha by AnchorDistance decile

Monthly calendar-time portfolios: a deal announced in month tau enters its decile portfolio for holding months tau+1 .. tau+12. Portfolio excess returns are regressed on Fama-French 5 factors plus momentum (Mkt-RF, SMB, HML, RMW, CMA, UMD) with Newey-West standard errors.

**Inputs (all already in `data/`):** the enriched SDC file (deal, acquirer PERMNO, `anchor_decile`), the acquirer daily CRSP files (`dlyret`, `dlycap`), the monthly French 5-factor file and the monthly momentum file.

**Data notes**
- Monthly acquirer returns are the compounded daily CRSP `dlyret` within each calendar month (non-missing days only). The CRSP v2 daily return is assumed to already include delisting returns; there is no separate `DLRET` column in the files, so `(1+RET)(1+DLRET)-1` is not applied.
- Value weights use the acquirer's month-end `dlycap` of month t-1 (CRSP market cap).
- Factors and returns are in percent. Alpha is % per month; annualized alpha = 12 x monthly alpha.
- Deal-weighted by default. The firm-deduplicated variant counts an acquirer once per decile-month even if it has several active deals in that decile.
- Series: D10 excess return (R_D10 - RF), D1 excess return (R_D1 - RF), and the long-short spread R_D1 - R_D10.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

pd.set_option("display.width", 250, "display.max_columns", 40)

ROOT = Path("..")
SDC_FILE = ROOT / "data/interim/sdc_pr_52w_high_decile.csv"
ACQ_FILE = lambda k: ROOT / f"data/interim/acquirer_decile_{k}.csv"
FF5_MONTHLY = ROOT / "data/external/F-F_Research_Data_5_Factors_2x3.csv"
MOM_MONTHLY = ROOT / "data/external/F-F_Momentum_Factor.csv"
RESULTS, PROCESSED = ROOT / "results", ROOT / "data/processed"

KEYS = ["master_cusip", "acusip", "dateann"]
HOLD = range(1, 13)                                   # holding months tau+1 .. tau+12
FACTORS = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "UMD"]

## 1. Factors (monthly, percent)

In [2]:
def read_french_monthly(path):
    """Monthly rows only (yyyymm). The file also holds annual rows (yyyy) and a footer, which are skipped."""
    rows, header = [], None
    for line in Path(path).read_text().splitlines():
        parts = [p.strip() for p in line.split(",")]
        if header is None and parts[0] == "" and len(parts) > 1 and parts[1]:
            header = parts[1:]
        elif re.fullmatch(r"\d{6}", parts[0]):
            rows.append([parts[0]] + parts[1:])
    df = pd.DataFrame(rows, columns=["ym"] + header).set_index("ym").astype(float)
    df.index = pd.PeriodIndex(df.index, freq="M")
    return df

ff = read_french_monthly(FF5_MONTHLY).join(read_french_monthly(MOM_MONTHLY).rename(columns={"Mom": "UMD"}), how="inner")
ff = ff[FACTORS + ["RF"]]
print(ff.index.min(), "to", ff.index.max(), "|", len(ff), "months")
print("missing values:", int(ff.isna().sum().sum()))
ff.tail(3)

1963-07 to 2026-09 | 759 months
missing values: 0


,Mkt-RF,SMB,HML,RMW,CMA,UMD,RF
ym,,,,,,,
2026-07,-0.61,-0.09,2.11,11.07,2.27,-12.21,0.33
2026-08,2.56,-0.52,-3.54,-4.09,-2.01,-5.60,0.29
2026-09,-1.07,-4.60,-1.55,-3.22,1.08,6.70,0.31


## 2. Deal-month returns for the 12 holding months
Daily acquirer returns are compounded into calendar months; each deal contributes months tau+1 .. tau+12.

In [3]:
sdc = pd.read_csv(SDC_FILE, dtype={"master_cusip": str, "acusip": str, "dateann": str}, low_memory=False)
deals = sdc.drop_duplicates(KEYS)
deals = deals.loc[deals["anchor_decile"].notna(), KEYS + ["anchor_decile"]].copy()
deals["anchor_decile"] = deals["anchor_decile"].astype(int)
deals["tau"] = pd.to_datetime(deals["dateann"]).dt.to_period("M")
print(len(deals), "deals; announced", deals["tau"].min(), "to", deals["tau"].max())

acq = pd.concat([pd.read_csv(ACQ_FILE(k), usecols=KEYS + ["permno", "dlycaldt", "dlyret", "dlycap"],
                             dtype={"master_cusip": str, "acusip": str, "dateann": str}) for k in range(1, 11)],
                ignore_index=True)
acq["dlycaldt"] = pd.to_datetime(acq["dlycaldt"])
acq["ym"] = acq["dlycaldt"].dt.to_period("M")
acq = acq.sort_values("dlycaldt")
print(f"{len(acq):,} daily rows, {acq['dlycaldt'].min().date()} to {acq['dlycaldt'].max().date()}")

1461 deals; announced 2005-01 to 2024-12


800,529 daily rows, 2003-12-01 to 2025-12-31


In [4]:
# daily -> monthly per deal: compounded return, month-end market cap
acq["lr"] = np.log1p(acq["dlyret"])
g = acq.groupby(KEYS + ["permno", "ym"])
monthly = g.agg(ret=("lr", lambda s: np.expm1(s.sum()) if s.notna().any() else np.nan),
                cap_end=("dlycap", "last"), n_days=("dlyret", "count")).reset_index()
monthly["m_ord"] = monthly["ym"].dt.year * 12 + monthly["ym"].dt.month

# lagged (t-1) month-end market cap: join each month to the previous month's cap_end
prev = monthly[KEYS + ["m_ord", "cap_end"]].rename(columns={"cap_end": "cap_lag"})
prev["m_ord"] += 1
monthly = monthly.merge(prev, on=KEYS + ["m_ord"], how="left")

dm = monthly.merge(deals, on=KEYS, how="inner")
dm["k"] = dm["m_ord"] - (dm["tau"].dt.year * 12 + dm["tau"].dt.month)      # months since announcement month
dm = dm[dm["k"].isin(HOLD) & dm["ret"].notna()].copy()
dm["ret"] *= 100                                                            # percent, like the factors
dm = dm.merge(ff[["RF"]], left_on="ym", right_index=True, how="inner")      # drops months after the factor file ends (none expected)
print(f"{len(dm):,} deal-months | {dm['ym'].min()} to {dm['ym'].max()}")
exp = len(deals) * 12
print(f"expected if every deal had all 12 months: {exp:,}; actual {len(dm):,} ({len(dm)/exp:.1%})")
print("deals with fewer than 12 months (acquirer delisted or data ended):",
      int((dm.groupby(KEYS).size() < 12).sum()))

17,440 deal-months | 2005-02 to 2025-12
expected if every deal had all 12 months: 17,532; actual 17,440 (99.5%)
deals with fewer than 12 months (acquirer delisted or data ended): 20


## 3. Decile portfolios by month
EW = equal-weighted mean of deal-month returns. VW = weighted by market cap at the end of t-1. `dedup` counts each acquirer once per decile-month.

In [5]:
def decile_portfolios(dm, dedup=False):
    d = dm.drop_duplicates(["anchor_decile", "permno", "ym"]) if dedup else dm
    rows = []
    for (dec, ym), x in d.groupby(["anchor_decile", "ym"]):
        v = x.dropna(subset=["cap_lag"])
        rows.append(dict(decile=dec, ym=ym, N=len(x), EW=x["ret"].mean(),
                         VW=np.average(v["ret"], weights=v["cap_lag"]) if len(v) else np.nan))
    return pd.DataFrame(rows)

def to_series(port, weighting, ff):
    """Monthly D1, D10 and long-short series with deal counts, merged with factors."""
    r = port.pivot(index="ym", columns="decile", values=weighting)
    n = port.pivot(index="ym", columns="decile", values="N")
    out = pd.DataFrame(index=ff.index.intersection(r.index.union(n.index)))
    out["R1"], out["R10"] = r[1].reindex(out.index), r[10].reindex(out.index)
    out["N1"], out["N10"] = n[1].reindex(out.index), n[10].reindex(out.index)
    return out.join(ff)

ports = {False: decile_portfolios(dm), True: decile_portfolios(dm, dedup=True)}
avgN = ports[False].groupby("decile")["N"].agg(["mean", "min", "max", "count"]).round(1)
avgN.columns = ["avg deals/month", "min", "max", "active months"]
avgN

,avg deals/month,min,max,active months
decile,,,,
1,7.2,1,26,244
2,6.9,1,20,251
3,7.0,1,16,249
4,6.9,1,16,251
5,7.0,1,24,248
6,6.9,1,16,250
7,7.1,1,20,247
8,7.4,1,18,238
9,7.1,1,14,248


## 4. Six-factor regressions
`y = alpha + b1 Mkt-RF + b2 SMB + b3 HML + b4 RMW + b5 CMA + b6 UMD + e`, OLS with Newey-West (HAC) standard errors. Specifications:

| Spec | Description |
|---|---|
| `NW3` | all active months, 3 lags (primary) |
| `NW6` | all active months, 6 lags |
| `N>=5`, `N>=10` | only months with at least 5 / 10 deals in the portfolio (for the spread, in both legs), 3 lags |
| `WLS` | weighted by sqrt(N_t), 3 lags |

A regression with fewer than 30 active months (7 parameters) is reported as missing, not estimated.

In [6]:
MIN_MONTHS = 30          # regressions with fewer months are reported as not estimable
SPECS = [("NW3", 3, 1, False), ("NW6", 6, 1, False), ("N>=5", 3, 5, False), ("N>=10", 3, 10, False), ("WLS", 3, 1, True)]

def build_y(s, which):
    if which == "D10":  return s["R10"] - s["RF"], s["N10"]
    if which == "D1":   return s["R1"] - s["RF"], s["N1"]
    return s["R1"] - s["R10"], np.fmin(s["N1"], s["N10"])               # long-short D1 - D10

def run_reg(s, which, lag, nmin, wls):
    y, n = build_y(s, which)
    d = pd.concat([y.rename("y"), n.rename("N"), s[FACTORS]], axis=1).dropna()
    d = d[d["N"] >= nmin]
    if len(d) < MIN_MONTHS:                                              # too few months for 7 parameters: not estimable
        return dict(months=len(d), alpha_m=np.nan, alpha_ann=np.nan, t=np.nan, p=np.nan, R2=np.nan,
                    **{f"b_{k}": np.nan for k in FACTORS}, **{f"t_{k}": np.nan for k in FACTORS})
    X = sm.add_constant(d[FACTORS])
    model = sm.WLS(d["y"], X, weights=np.sqrt(d["N"])) if wls else sm.OLS(d["y"], X)
    f = model.fit(cov_type="HAC", cov_kwds={"maxlags": lag})
    row = dict(months=int(f.nobs), alpha_m=f.params["const"], alpha_ann=12 * f.params["const"],
               t=f.tvalues["const"], p=f.pvalues["const"])
    row.update({f"b_{k}": f.params[k] for k in FACTORS})
    row.update({f"t_{k}": f.tvalues[k] for k in FACTORS})
    row["R2"] = f.rsquared
    return row

rows = []
for dedup in (False, True):
    for w in ("EW", "VW"):
        s = to_series(ports[dedup], w, ff)
        for which in ("D10", "D1", "D1-D10"):
            for name, lag, nmin, wls in SPECS:
                rows.append(dict(weighting=w, dedup=dedup, series=which, spec=name) | run_reg(s, which, lag, nmin, wls))
reg = pd.DataFrame(rows)
print(len(reg), "regressions")

60 regressions


### Primary results: deal-weighted, all active months, Newey-West 3 lags

In [7]:
def stars(p): return "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""
show = ["weighting", "series", "months", "alpha_m", "alpha_ann", "t", "p"] + [f"b_{k}" for k in FACTORS] + ["R2"]
prim = reg[(reg.spec == "NW3") & (~reg.dedup)][show].copy()
prim["sig"] = prim["p"].map(stars)
print(prim.round(3).to_string(index=False))

weighting series  months  alpha_m  alpha_ann      t     p  b_Mkt-RF  b_SMB  b_HML  b_RMW  b_CMA  b_UMD    R2 sig
       EW    D10     241   -0.766     -9.196 -2.347 0.019     0.954  0.526  0.006 -0.087  0.306  0.135 0.442  **
       EW     D1     244   -0.217     -2.609 -0.505 0.614     1.104  1.211  0.343 -0.525 -0.297 -0.220 0.522    
       EW D1-D10     237    0.568      6.821  1.074 0.283     0.154  0.696  0.363 -0.392 -0.580 -0.452 0.149    
       VW    D10     241   -0.540     -6.481 -1.590 0.112     0.955 -0.072 -0.023  0.091  0.308  0.138 0.291    
       VW     D1     244    0.244      2.927  0.539 0.590     0.889  0.389  0.337 -0.413 -0.014 -0.349 0.420    
       VW D1-D10     237    0.927     11.123  1.803 0.071    -0.173  0.582  0.134 -0.412 -0.122 -0.539 0.119   *


### Robustness: other lags, minimum deal counts, WLS, and firm-deduplicated portfolios

In [8]:
rob = reg[["dedup", "weighting", "series", "spec", "months", "alpha_m", "alpha_ann", "t", "p", "b_UMD", "R2"]].copy()
rob["sig"] = rob["p"].map(stars)
for series in ("D10", "D1", "D1-D10"):
    print(f"--- {series}: alpha (% per month), t-stat, months")
    x = rob[rob.series == series]
    x = x.assign(cell=x.apply(lambda r: f"{r.alpha_m:.2f}{r.sig} (t={r.t:.2f}, n={r.months})", axis=1))
    print(x.pivot(index=["dedup", "weighting"], columns="spec", values="cell")[[s[0] for s in SPECS]].to_string(), "\n")

--- D10: alpha (% per month), t-stat, months
spec                                  NW3                       NW6                      N>=5                    N>=10                        WLS
dedup weighting                                                                                                                                  
False EW         -0.77** (t=-2.35, n=241)  -0.77** (t=-2.30, n=241)  -0.44** (t=-2.20, n=164)    -0.24 (t=-0.77, n=73)  -0.60*** (t=-2.61, n=241)
      VW           -0.54 (t=-1.59, n=241)   -0.54* (t=-1.71, n=241)    -0.49 (t=-1.61, n=164)  -0.86** (t=-2.02, n=73)    -0.54* (t=-1.77, n=241)
True  EW         -0.78** (t=-2.38, n=241)  -0.78** (t=-2.34, n=241)  -0.45** (t=-2.28, n=164)    -0.24 (t=-0.78, n=65)  -0.60*** (t=-2.64, n=241)
      VW           -0.54 (t=-1.60, n=241)   -0.54* (t=-1.72, n=241)    -0.49 (t=-1.60, n=164)   -0.83* (t=-1.92, n=65)    -0.53* (t=-1.76, n=241) 

--- D1: alpha (% per month), t-stat, months
spec                             

## 5. Save
The regression table is aggregate and goes to `results/`. The monthly portfolio series (derived from CRSP) go to `data/processed/` and are not tracked.

## 5. Pooled groups: D2-3 vs D8-9
Same portfolios, regressions and specifications, with deciles pooled: the low group is D2 and D3, the high group is D8 and D9 (long low, short high). In the output below the low group is labelled `Low` and the high group `High`; the spread is `Low-High`.

In [9]:

LOW, HIGH = [2, 3], [8, 9]
grp = dm[dm["anchor_decile"].isin(LOW + HIGH)].copy()
grp["anchor_decile"] = np.where(grp["anchor_decile"].isin(LOW), 1, 10)      # 1 = Low group, 10 = High group

gports = {False: decile_portfolios(grp), True: decile_portfolios(grp, dedup=True)}
print("avg deals per month  Low:", round(gports[False].query("decile == 1")["N"].mean(), 1),
      "| High:", round(gports[False].query("decile == 10")["N"].mean(), 1))

rows = []
for dedup in (False, True):
    for w in ("EW", "VW"):
        s = to_series(gports[dedup], w, ff)
        for which, label in [("D10", "High"), ("D1", "Low"), ("D1-D10", "Low-High")]:
            for name, lag, nmin, wls in SPECS:
                rows.append(dict(weighting=w, dedup=dedup, series=label, spec=name) | run_reg(s, which, lag, nmin, wls))
greg = pd.DataFrame(rows)

show = ["weighting", "series", "months", "alpha_m", "alpha_ann", "t", "p"] + [f"b_{k}" for k in FACTORS] + ["R2"]
prim = greg[(greg.spec == "NW3") & (~greg.dedup)][show].copy()
prim["sig"] = prim["p"].map(stars)
print("Primary (deal-weighted, NW 3 lags)")
print(prim.round(3).to_string(index=False))


avg deals per month  Low: 13.8 | High: 14.1
Primary (deal-weighted, NW 3 lags)
weighting   series  months  alpha_m  alpha_ann      t     p  b_Mkt-RF  b_SMB  b_HML  b_RMW  b_CMA  b_UMD    R2 sig
       EW     High     248    0.092      1.101  0.514 0.607     0.883  0.306  0.303 -0.101  0.044 -0.091 0.762    
       EW      Low     251   -0.206     -2.472 -1.119 0.263     0.904  0.743  0.110 -0.012  0.039 -0.089 0.785    
       EW Low-High     248   -0.217     -2.604 -0.866 0.387     0.017  0.411 -0.174  0.014 -0.061  0.002 0.076    
       VW     High     248   -0.246     -2.947 -1.334 0.182     0.875  0.028 -0.019  0.043  0.455  0.002 0.608    
       VW      Low     251   -0.544     -6.533 -1.867 0.062     0.887  0.056 -0.055 -0.142  0.122 -0.011 0.496   *
       VW Low-High     248   -0.201     -2.406 -0.615 0.539     0.010 -0.013 -0.017 -0.299 -0.407 -0.013 0.046    


In [10]:

rob = greg[["dedup", "weighting", "series", "spec", "months", "alpha_m", "t", "p"]].copy()
rob["sig"] = rob["p"].map(stars)
for series in ("High", "Low", "Low-High"):
    print(f"--- {series}: alpha (% per month), t-stat, months")
    x = rob[rob.series == series]
    x = x.assign(cell=x.apply(lambda r: f"{r.alpha_m:.2f}{r.sig} (t={r.t:.2f}, n={r.months})", axis=1))
    print(x.pivot(index=["dedup", "weighting"], columns="spec", values="cell")[[s[0] for s in SPECS]].to_string(), "\n")


--- High: alpha (% per month), t-stat, months
spec                                NW3                      NW6                    N>=5                    N>=10                     WLS
dedup weighting                                                                                                                          
False EW           0.09 (t=0.51, n=248)     0.09 (t=0.54, n=248)    0.20 (t=1.00, n=232)     0.09 (t=0.49, n=173)    0.13 (t=0.79, n=248)
      VW         -0.25 (t=-1.33, n=248)   -0.25 (t=-1.54, n=248)  -0.21 (t=-1.15, n=232)  -0.28* (t=-1.68, n=173)  -0.22 (t=-1.32, n=248)
True  EW           0.08 (t=0.47, n=248)     0.08 (t=0.49, n=248)    0.19 (t=0.96, n=232)     0.07 (t=0.42, n=173)    0.12 (t=0.73, n=248)
      VW         -0.26 (t=-1.45, n=248)  -0.26* (t=-1.68, n=248)  -0.23 (t=-1.27, n=232)  -0.30* (t=-1.84, n=173)  -0.24 (t=-1.50, n=248) 

--- Low: alpha (% per month), t-stat, months
spec                                 NW3                      NW6              

## 6. Decile 10 by form of payment
Is the Decile 10 alpha present in all-cash deals, or only where stock is part of the payment? Payment form comes from SDC `considoff` (components) and `consid` (per-share terms). Option and warrant cash-outs and assumed liabilities are ignored: they are not the consideration paid for the target's shares.

- **Cash**: cash and no stock. **Stock**: stock and no cash. **Mixed**: both. **Other**: neither (for example only notes or contingent value rights).

As a control, the same split is run on deciles D1-D9. If stock deals underperform everywhere, a stock effect in D10 is not specific to anchoring.

*Not tested: contested auctions.* The SDC pull has no competing-bidder variable (see the notes at the end of this section).

In [11]:

STOCK_TOK = ("Common Stock", "Ordinary Share", "Preference Share", "Preferred", "Convert. Preferred", "Tracking Stock",
             "Third Party Com Shs", "Common L.P. Unit", "Unit")
CASH_TOK = ("Cash", "Cash Only", "Cash Dividend")

def payment_form(considoff, consid):
    toks = [t.strip() for t in str(considoff).split(",")]
    stock = any(t.startswith(STOCK_TOK) for t in toks)
    # per-share clauses of the SDC description; clauses with an aggregate "mil" amount are option/warrant/liability payouts
    clauses = [c for c in str(consid).split(";") if " mil" not in c.lower()] or [str(consid)]
    main = " ".join(clauses).lower()
    cash = ("Cash Only" in toks) or ("cash" in main) or (not stock and any(t in CASH_TOK for t in toks))
    return "Mixed" if stock and cash else "Stock" if stock else "Cash" if cash else "Other"

pay = sdc.drop_duplicates(KEYS)[KEYS + ["consid", "considoff"]].copy()
pay["pay"] = [payment_form(a, b) for a, b in zip(pay["considoff"], pay["consid"])]
d_pay = deals.merge(pay[KEYS + ["pay", "consid", "considoff"]], on=KEYS, how="left")
print(pd.crosstab(d_pay["anchor_decile"].where(d_pay["anchor_decile"] == 10, 0).map({10: "D10", 0: "D1-D9"}), d_pay["pay"], margins=True))


pay            Cash  Mixed  Stock   All
anchor_decile                          
D1-D9           555    371    389  1315
D10              81     35     30   146
All             636    406    419  1461


In [12]:

# eyeball check of the labels on Decile 10: a few examples of each class
chk = d_pay[d_pay["anchor_decile"] == 10]
for lab in ["Cash", "Stock", "Mixed", "Other"]:
    print(f"--- {lab}")
    for _, r in chk[chk["pay"] == lab].head(5).iterrows():
        print("   ", r["considoff"], "|", str(r["consid"])[:110])


--- Cash
    Cash Only, Cash Only | $27.33/sh com; $0.400 mil/ options
    Cash Only, Cash Only | $22.50 cash/sh com; plus $89. 753 mil/options
    Cash, Cash | A sweetened $36.5/sh com; plus $54.055 mil/options; originally, $35/sh com; plus $50.099 mil/options
    Cash Only, Cash Only | $24.14 cash/sh com; plus $2. 374 mil/options
    Cash Only, Cash Only | $24.8 cash/sh com; plus $9. 655 mil/options
--- Stock
    Common Stock | 0.68 shs com/sh com; plus $12.473 mil/options
    Common Stock, Liabilities | A sweetened .82 sh com/com plus USD 55.096 mil/opt; originally, .80 sh com/sh com plus USD 90.36 mil/options
    Common Stock | A sweetened .54 shs com/sh com; originally .461 shs com/ sh com
    Common Stock, Liabilities | 0.08 shs com/sh com; plus the assumption of an estimated $6.2 mil in a liabilities
    Common Stock | An estimated USD 94.5 mil com/ com subject to a collar agreement of 32.5% and 58% of Celldex's outstanding com
--- Mixed
    Cash, Common Stock | A choice of $24.

In [13]:

dm_pay = dm.merge(pay[KEYS + ["pay"]], on=KEYS, how="left")

def one_group_series(sub, weighting, dedup=False):
    """Monthly series for a single group of deals (treated as 'D10' in run_reg)."""
    s = sub.assign(anchor_decile=10)
    port = decile_portfolios(s, dedup=dedup)
    r = port.set_index("ym")
    out = pd.DataFrame(index=ff.index.intersection(r.index))
    out["R10"], out["N10"] = r[weighting].reindex(out.index), r["N"].reindex(out.index)
    out["R1"], out["N1"] = np.nan, np.nan
    return out.join(ff)

groups = {
    "D10 all":            dm_pay[dm_pay.anchor_decile == 10],
    "D10 cash":           dm_pay[(dm_pay.anchor_decile == 10) & (dm_pay.pay == "Cash")],
    "D10 any stock":      dm_pay[(dm_pay.anchor_decile == 10) & dm_pay.pay.isin(["Stock", "Mixed"])],
    "D10 stock only":     dm_pay[(dm_pay.anchor_decile == 10) & (dm_pay.pay == "Stock")],
    "D10 mixed":          dm_pay[(dm_pay.anchor_decile == 10) & (dm_pay.pay == "Mixed")],
    "D1-9 cash (control)":      dm_pay[(dm_pay.anchor_decile < 10) & (dm_pay.pay == "Cash")],
    "D1-9 any stock (control)": dm_pay[(dm_pay.anchor_decile < 10) & dm_pay.pay.isin(["Stock", "Mixed"])],
}
rows = []
for gname, sub in groups.items():
    ndeals = sub[KEYS].drop_duplicates().shape[0]
    for w in ("EW", "VW"):
        s = one_group_series(sub, w)
        for name, lag, nmin, wls in SPECS:
            rows.append(dict(group=gname, deals=ndeals, avg_N=round(s["N10"].mean(), 1), weighting=w, spec=name)
                        | run_reg(s, "D10", lag, nmin, wls))
preg = pd.DataFrame(rows)
cols = ["group", "deals", "avg_N", "weighting", "months", "alpha_m", "alpha_ann", "t", "p", "b_UMD", "R2"]
prim = preg[preg.spec == "NW3"][cols].copy(); prim["sig"] = prim["p"].map(stars)
print("Primary (deal-weighted, NW 3 lags); alpha in % per month")
print(prim.round(3).to_string(index=False))


Primary (deal-weighted, NW 3 lags); alpha in % per month
                   group  deals  avg_N weighting  months  alpha_m  alpha_ann      t     p  b_UMD    R2 sig
                 D10 all    146    7.3        EW     241   -0.766     -9.196 -2.347 0.019  0.135 0.442  **
                 D10 all    146    7.3        VW     241   -0.540     -6.481 -1.590 0.112  0.138 0.291    
                D10 cash     81    4.2        EW     234   -0.828     -9.935 -2.773 0.006  0.063 0.505 ***
                D10 cash     81    4.2        VW     234   -0.677     -8.123 -1.930 0.054  0.010 0.354   *
           D10 any stock     65    3.6        EW     215   -0.609     -7.306 -1.082 0.279  0.174 0.318    
           D10 any stock     65    3.6        VW     215   -0.472     -5.667 -0.801 0.423  0.089 0.292    
          D10 stock only     30    2.0        EW     178   -1.461    -17.533 -2.277 0.023  0.250 0.330  **
          D10 stock only     30    2.0        VW     178   -1.492    -17.905 -2.180 0.0

In [14]:

x = preg[["group", "weighting", "spec", "months", "alpha_m", "t", "p"]].copy()
x["sig"] = x["p"].map(stars)
x["cell"] = x.apply(lambda r: f"{r.alpha_m:.2f}{r.sig} (t={r.t:.2f}, n={r.months})", axis=1)
for w in ("EW", "VW"):
    print(f"--- {w}: alpha (% per month), t-stat, months")
    print(x[x.weighting == w].pivot(index="group", columns="spec", values="cell")[[s[0] for s in SPECS]]
          .loc[list(groups)].to_string(), "\n")


--- EW: alpha (% per month), t-stat, months
spec                                            NW3                        NW6                      N>=5                   N>=10                        WLS
group                                                                                                                                                      
D10 all                    -0.77** (t=-2.35, n=241)   -0.77** (t=-2.30, n=241)  -0.44** (t=-2.20, n=164)   -0.24 (t=-0.77, n=73)  -0.60*** (t=-2.61, n=241)
D10 cash                  -0.83*** (t=-2.77, n=234)  -0.83*** (t=-2.87, n=234)  -1.06*** (t=-2.65, n=94)        nan (t=nan, n=4)  -0.81*** (t=-3.05, n=234)
D10 any stock                -0.61 (t=-1.08, n=215)     -0.61 (t=-1.06, n=215)     -0.11 (t=-0.26, n=62)       nan (t=nan, n=13)     -0.36 (t=-0.91, n=215)
D10 stock only             -1.46** (t=-2.28, n=178)   -1.46** (t=-2.29, n=178)         nan (t=nan, n=13)        nan (t=nan, n=0)   -1.21** (t=-2.21, n=178)
D10 mixed           

**Contested auctions: not testable with the current data.** The SDC pull has no competing-bidder, hostile or tender-offer variable, and the sample keeps only completed deals, so losing bids are not in the file. Counting several deals for one target would not identify bidding wars. To test it, add the competing-bid fields to the SDC query (check the column list of `tr_sdc_ma.wrds_ma_details` for a competing-bid flag, a hostile or unsolicited flag, and a tender-offer flag), re-pull, and merge on the deal key.

## 7. Save
The regression tables are aggregate and go to `results/`. The monthly portfolio series (derived from CRSP) go to `data/processed/` and are not tracked.

In [15]:
RESULTS.mkdir(exist_ok=True); PROCESSED.mkdir(parents=True, exist_ok=True)
reg.to_csv(RESULTS / "calendar_time_six_factor_alpha.csv", index=False)
greg.to_csv(RESULTS / "calendar_time_six_factor_alpha_d23_vs_d89.csv", index=False)
preg.to_csv(RESULTS / "calendar_time_six_factor_alpha_d10_by_payment.csv", index=False)
out = []
for dedup in (False, True):
    for w in ("EW", "VW"):
        out.append(to_series(ports[dedup], w, ff).assign(weighting=w, dedup=dedup).reset_index(names="ym"))
pd.concat(out).to_csv(PROCESSED / "calendar_time_portfolio_returns.csv", index=False)